# Negative Prices in Linearized Unit Commitment

*Adapted from: [PyPSA Examples &ndash; Negative Prices in Linearized Unit Commitment](https://docs.pypsa.org/latest/examples/uc-prices/)*

In [1]:
import datetime as dt

import pandas as pd

import typsa
from typsa.components import Bus, Carrier, CommittableGenerator, Load
from typsa.time_variation import IntegerSnapshots, RangedSeries

In [2]:
network = typsa.Network(IntegerSnapshots(range(5), spacing=dt.timedelta(hours=1)))

network.add_components(Carrier(name="AC"))
network.add_components(bus := Bus(name="bus", carrier="AC"))
network.add_components(
    load := Load(
        name="load",
        bus=bus.name,
        p_set=RangedSeries(pd.Series([50, 120, 50, 20, 50])),
    )
)
base_generator_marginal_cost = 20
network.add_components(
    base_generator := CommittableGenerator(
        name="base",
        bus=bus.name,
        p_nom=100,
        marginal_cost=base_generator_marginal_cost,
        p_min_pu=0.4,
        start_up_cost=4000,
        shut_down_cost=2000,
    ),
    peak_generator := CommittableGenerator(
        name="peak",
        bus=bus.name,
        p_nom=50,
        marginal_cost=70,
        p_min_pu=0.2,
        start_up_cost=250,
    ),
)

In [3]:
optimized_network, _ = network.optimize(linearized_unit_commitment=True)

INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.02s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 40 primals, 45 duals
Objective: 7.90e+03
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-com-p-lower, Generator-com-p-upper, Generator-com-transition-start-up, Generator-com-transition-shut-down, Generator-com-status-min_up_time_must_stay_up were not assigned to the network.


Running HiGHS 1.12.0 (git hash: 755a8e0): Copyright (c) 2025 HiGHS under MIT licence terms
LP linopy-problem-my68zmut has 45 rows; 40 cols; 106 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 1e+02]
  Cost    [2e+01, 4e+03]
  Bound   [1e+00, 1e+00]
  RHS     [1e+00, 1e+02]
Presolving model
38 rows, 28 cols, 87 nonzeros  0s
29 rows, 26 cols, 70 nonzeros  0s
Presolve reductions: rows 29(-16); columns 26(-14); nonzeros 70(-36) 
Solving the presolved LP
Using EKK dual simplex solver - serial
  Iteration        Objective     Infeasibilities num(sum)
          0    -8.0000000000e+05 Ph1: 16(23992); Du: 4(800) 0s
         10     7.9000000000e+03 Pr: 0(0) 0s

Performed postsolve
Solving the original LP from the solution after postsolve

Model name          : linopy-problem-my68zmut
Model status        : Optimal
Simplex   iterations: 10
Objective value     :  7.9000000000e+03
P-D objective error :  0.0000000000e+00
HiGHS run time      :          0.00


In [4]:
dispatch = optimized_network.dynamic_results.of_all_generators.p
status = optimized_network.dynamic_results.of_committable_generators.status
pd.DataFrame(
    {
        "Load (MW)": optimized_network.dynamic_results.of_all_loads.p[load.name],
        "Base Gen (MW)": dispatch[base_generator.name],
        "Peak Gen (MW)": dispatch[peak_generator.name],
        "Total Gen (MW)": dispatch.sum(axis="columns"),
        "Base Status": status[base_generator.name],
        "Peak Status": status[peak_generator.name],
        "Price (€/MWh)": optimized_network.dynamic_results.of_all_buses.marginal_price[
            bus.name
        ],
    }
).rename_axis("Time Period")

,Load (MW),Base Gen (MW),Peak Gen (MW),Total Gen (MW),Base Status,Peak Status,Price (€/MWh)
Time Period,,,,,,,
0,50.0,50.0,-0.0,50.0,1.0,0.0,20.0
1,120.0,100.0,20.0,120.0,1.0,0.4,75.0
2,50.0,50.0,-0.0,50.0,1.0,0.0,20.0
3,20.0,20.0,-0.0,20.0,0.5,0.0,-30.0
4,50.0,50.0,-0.0,50.0,0.5,0.0,20.0


In [5]:
periods_low = [2, 3, 4]

cycle_cost = base_generator.start_up_cost + base_generator.shut_down_cost

gen_low = float(dispatch.loc[periods_low, base_generator.name].sum())
op_cost = gen_low * base_generator_marginal_cost

print("Why stay online?")
print("=" * 25)
print(f"Start-up cost:        {base_generator.start_up_cost:,.0f} €")
print(f"Shut-down cost:       {base_generator.shut_down_cost:,.0f} €")
print(f"Total cycling cost:   {cycle_cost:,.0f} €\n")

print(f"Output (snapshots 2-4): {gen_low:.1f} MWh")
print(f"Operational cost:       {op_cost:,.0f} €\n")

decision = "Stay online" if op_cost < cycle_cost else "Cycle off/on"
savings = abs(cycle_cost - op_cost)

print(f"Decision: {decision} is cheaper.")
print(f"Savings vs alternative: {savings:,.0f} €")

Why stay online?
Start-up cost:        4,000 €
Shut-down cost:       2,000 €
Total cycling cost:   6,000 €

Output (snapshots 2-4): 120.0 MWh
Operational cost:       2,400 €

Decision: Stay online is cheaper.
Savings vs alternative: 3,600 €
